# 08 · Phase B (vLLM): query-side LLM work, LLM methods and agentic runs

Plan sections 4.4, 4.5, 4.7 and 5 (phase B). vLLM holds the GPU; the encoder runs on the **CPU** and only embeds short texts (HyDE passages, agent search queries).

1. **Query-side LLM work** for all 100 questions: HyDE passages (Q1), keyword expansions (Q2) and the engine-type filter (F1). Qwen3.8-27B-INT4, thinking off, temperature 0, JSON-schema output where JSON is needed. Every call is cached in `data/cache/llm.jsonl` under the hash of the full request. The outputs don't depend on the index.
2. **Runs that need the LLM**: Q1, Q2, F1 (gate off and on) and T4, the agent (gate on, 4k / 8k / 16k), on both indexes.
3. **Verification** of all runs (plan 8): reproducibility, no label leakage, one hop.

> **Before running: swap from phase A to phase B.**
> ```bash
> docker compose stop encoder-gpu                  # frees the GPU (and port 8081)
> docker compose up -d qdrant vllm && docker compose up -d --force-recreate encoder         # CPU encoder + vLLM (VLLM_CONFIG=serve.retrieval.yaml)
> docker compose logs -f vllm                      # ~2.5 min to healthy
> ```
> No reranker runs here: X1 and X2 ran in notebook 07 on the GPU encoder.

Times: the query-side calls take ~5–15 s per question; the agent takes ~1–2 h for the six T4 runs.

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # all paths below are relative to the repo root
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [2]:
import json, re, subprocess, time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from csrag import runs
from csrag.corpus import COLLECTIONS, Corpus
from csrag.retrievers import Pipeline, Resources, grid, method_config, prompt_hashes
from csrag.retrievers.query_llm import vllm_healthy

PARAMS_VERSION = "v1"

t0 = time.time()
while not vllm_healthy():
    if time.time() - t0 > 900:
        raise TimeoutError("vLLM not healthy: `docker compose up -d vllm`, then `docker compose logs vllm`")
    time.sleep(10)
print("vLLM healthy after", round(time.time() - t0), "s")
corpus = Corpus(DATA)
res = Resources.connect(corpus, DATA)
health = res.encoder.wait_ready()
print("encoder on", health["device"], "| LLM", res.llm.model)
assert health["device"] == "cpu", "phase B uses the CPU encoder (`docker compose up -d encoder`); the GPU belongs to vLLM"

vLLM healthy after 0 s
encoder on cpu | LLM RedHatAI/Qwen3.8-27B-INT4


/home/goksu/projects/ikincitik/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:287: UserWarning: Qdrant client version 1.20.0 is incompatible with server version 1.18.3. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


### KV-cache check (plan 3.1)

The startup log must show a KV cache that holds at least one full 32,768-token request. If it doesn't, set `max-model-len: 24576` in `vllm/serve.retrieval.yaml` and drop the 16k expansion budget.

In [3]:
try:
    log = subprocess.run(["docker", "logs", "ikincitik-vllm"], capture_output=True, text=True, timeout=60)
    text = log.stdout + log.stderr
    conc = [float(x) for x in re.findall(r"Maximum concurrency for 32,768 tokens per request: ([\d.]+)x", text)]
    kv = re.findall(r"GPU KV cache size: ([\d,]+) tokens", text)
    print("KV cache tokens:", kv[-1] if kv else "?", "| max concurrency at 32768:", conc[-1] if conc else "?")
    assert conc and conc[-1] >= 1.0, "KV cache does not hold one 32768-token request"
except FileNotFoundError:
    print("docker CLI not available here: check `docker compose logs vllm | grep concurrency` by hand")

KV cache tokens: 42,325 | max concurrency at 32768: 1.29


In [4]:
questions = [json.loads(l) for l in open(DATA / "eval" / "questions.jsonl", encoding="utf-8")]
texts = [q["question"] for q in questions]
prompts = prompt_hashes()
configs = grid(PARAMS_VERSION, prompts=prompts)
print(res.llm.model, "| prompts:", prompts)

RedHatAI/Qwen3.8-27B-INT4 | prompts: {'hyde': '540ea872', 'keywords': '6e983fae', 'filter': '1ded7ff7', 'agent': '6e9911bc'}


## 1. Query-side LLM work

In [5]:
rows = []
for q in tqdm(questions):
    passage, r1 = res.llm.hyde(q["question"])
    terms, r2 = res.llm.keywords(q["question"])
    engine, r3 = res.llm.extract_filter(q["question"])
    rows.append({"question_id": q["question_id"], "question": q["question"], "hyde": passage, "terms": terms,
                 "f1_engine_type": engine, "label_engine_type": q["engine_type"], "answerable": q["answerable"],
                 "llm_seconds": r1["seconds"] + r2["seconds"] + r3["seconds"],
                 "prompt_tokens": sum(r["usage"]["prompt_tokens"] for r in (r1, r2, r3)),
                 "completion_tokens": sum(r["usage"]["completion_tokens"] for r in (r1, r2, r3)),
                 "finish": [r["finish_reason"] for r in (r1, r2, r3)]})
out = pd.DataFrame(rows)
print(len(res.llm.cache), "cached LLM calls")
out[["llm_seconds", "prompt_tokens", "completion_tokens"]].describe().loc[["mean", "50%", "max"]]

  0%|          | 0/100 [00:00<?, ?it/s]

300 cached LLM calls


,llm_seconds,prompt_tokens,completion_tokens
mean,18.54936,411.55,314.63
50%,17.96900,410.50,304.50
max,30.66900,481.00,526.00


In [6]:
truncated = out[out["finish"].apply(lambda f: "length" in f)]
print(len(truncated), "questions with an output cut at max_tokens")
pd.set_option("display.max_colwidth", 200)
out[["question_id", "question", "hyde", "terms"]].head(5)

3 questions with an output cut at max_tokens


,question_id,question,hyde,terms
0,q001,"Besides the acceptable means of compliance in Book 2 of the engine certification specifications, which other document may also provide acceptable means of compliance to the Book 1 specifications?","The acceptable means of compliance (AMC) to the requirements of CS-E, Book 1, are provided in the corresponding AMC sections of CS-E, Book 2. In addition, the acceptable means of compliance to the...","[Book 2, Book 1, AMC, Acceptable Means of Compliance, CS-E, Engine Certification, Specification, Regulatory, Compliance, Means]"
1,q002,Which subparts of the engine certification specifications apply to turbine engines?,CS-E1.1 Scope. This Part applies to the certification of aircraft engines and their components. The subparts of this Part are structured to address specific engine types and operational requiremen...,"[CS-E2, CS-E3, CS-E4, CS-E5, CS-E6, CS-E7, CS-E8, CS-E9, CS-E10, CS-E11]"
2,q003,"In the terminology for engine critical parts, how is 'Damage Tolerance' defined?","For the purposes of CS-E, ""Damage Tolerance"" is defined as the ability of a structure to sustain a specified level of damage, such as a crack, without failing. This concept is fundamental to the e...","[damage tolerance analysis, critical structure, fracture mechanics, inspection interval, failure probability, leak before break, structural integrity, probabilistic assessment, detection probabili..."
3,q004,"What examples are given of aircraft-supplied resources that an Electronic Engine Control System (EECS) may depend on, and who is responsible for specifying the EECS requirements for them?","AMC CS-E.11.15(a)\n\nExamples of aircraft-supplied resources upon which an Electronic Engine Control System (EECS) may depend include electrical power, pneumatic air, hydraulic fluid, and data lin...","[aircraft-provided resources, aircraft-supplied data, aircraft interface, aircraft system, aircraft manufacturer, engine manufacturer, interface requirements, data link, control inputs, operating ..."
4,q005,What details must the servicing information in an engine's Instructions for Continued Airworthiness cover?,"The Instructions for Continued Airworthiness (ICA) shall contain the information necessary to maintain the engine in a safe condition. This information shall include, but not be limited to, the fo...","[maintenance, inspection, repair, modification, airworthiness limitations, time limits, life limits, scheduled maintenance, unscheduled maintenance, component replacement]"


### F1 extraction vs the `engine_type` label (evaluation only)

F1 never sees the label. The label is only compared here, and the oracle F0 is the one method that uses it. `all` questions should map to `null` (no filter); a wrong `piston`/`turbine` filter removes the gold section from every list.

In [7]:
pd.crosstab(out["label_engine_type"], out["f1_engine_type"].fillna("null (no filter)"), margins=True)

f1_engine_type,null (no filter),piston,turbine,All
label_engine_type,,,,
all,15,0,11,26
piston,0,24,0,24
turbine,4,0,46,50
All,19,24,57,100


HyDE passages in dense form (CPU encoder; ~100 passages of ~200 tokens):

In [8]:
t0 = time.time()
h_emb = res.qemb["hyde"].get(list(out["hyde"]), res.encoder)
print(len(h_emb), "HyDE embeddings in", round(time.time() - t0, 1), "s")

100 HyDE embeddings in 25.9 s


## 2. Runs that need the LLM

Q1, Q2 and F1 with the gate off and on (their LLM outputs come from the cache above), then the agent T4 (gate on, three budgets, both indexes). Each agent step is cached; the agent's `search` embeds its new queries with the CPU encoder.

In [9]:
def execute(cfgs, desc):
    ids = []
    for cfg in tqdm(cfgs, desc=desc):
        ids.append(runs.execute(cfg, Pipeline(cfg, res).run, questions, res.meta(cfg["index"])))
    return ids

t0 = time.time()
llm_ids = execute([c for c in configs if c["method"] in ("Q1", "Q2", "F1")], "Q1 Q2 F1")
print(len(llm_ids), "runs in", round(time.time() - t0), "s")

Q1 Q2 F1:   0%|          | 0/12 [00:00<?, ?it/s]

12 runs in 2 s


In [10]:
agent_cfgs = [c for c in configs if c["method"] == "T4"]
t0 = time.time()
agent_ids = execute(agent_cfgs, "agent")
print(len(agent_ids), "agent runs in", round((time.time() - t0) / 60), "min")
paths = pd.DataFrame([{"run_id": r, "question_id": s["question_id"], "path": " → ".join(s["agent"]["path"])}
                      for r in agent_ids for s in runs.load_stats(r)])
paths.groupby("run_id")["path"].value_counts().groupby(level=0).head(5).rename("questions").to_frame()

agent:   0%|          | 0/6 [00:00<?, ?it/s]

6 agent runs in 134 min


questions
run_id                            path                                                         
T4-b16k-units-gateon-v1-f395387b  stop                                                       92
                                  search → search → search → search → search → limit          3
                                  search → expand → search → search → search → limit          2
                                  search → search → search → search → stop                    1
                                  search → search → search → expand → search → limit          1
T4-b16k-win500-gateon-v1-3dc50e24 stop                                                       90
                                  search → search → search → search → search → limit          7
                                  expand → stop                                               1
                                  expand → expand → expand → stop                             1
                                  search → search → search → stop                             1
T4-b4k-units-gateon-v1-cda5c5ae   stop                                                       92
                                  search → search → search → stop                             3
                                  search → search → search → search → stop                    2
                                  expand → search → stop                                      1
                                  search → expand → search → search → stop                    1
T4-b4k-win500-gateon-v1-673c0907  stop                                                       90
                                  search → search → search → stop                             4
                                  expand → expand → expand → expand → search → limit          1
                                  expand → expand → stop                                      1
                                  search → search → stop                                      1
T4-b8k-units-gateon-v1-34a7b4ae   stop                                                       91
                                  search → search → search → search → search → limit          3
                                  search → search → search → stop                             2
                                  search → search → search → search → stop                    2
                                  expand → expand → stop                                      1
T4-b8k-win500-gateon-v1-8ace188b  stop                                                       90
                                  search → search → search → search → search → limit          4
                                  expand → stop                                               1
                                  expand → expand → stop                                      1
                                  search → search → search → search → stop                    1

## 3. Verification (plan 8)

* **Reproducibility**: re-running a config reproduces the same `hits.parquet` (temperature 0, cached LLM outputs). H1 needs no LLM; T4 replays its cached steps.
* **No label leakage**: no retriever except F0 reads a label; the interface only receives `question_id` and `text`.
* **One hop**: no context in T3, T2+T3 or T4 holds hop-2 content, and every hop-1 item's source is hop-0 content of the same question.
* **Completeness**: every configuration of the grid has a run (110 for a params version).

In [11]:
import tempfile
for cfg in [method_config("H1", "win500", True, PARAMS_VERSION, prompts=prompts),
            method_config("T4", "units", True, PARAMS_VERSION, budget=8192, prompts=prompts)]:
    rid = runs.run_id_of(cfg)
    with tempfile.TemporaryDirectory() as tmp:
        runs.execute(cfg, Pipeline(cfg, res).run, questions, res.meta(cfg["index"]), root=tmp)
        same = runs.load_hits(rid, tmp).equals(runs.load_hits(rid))
    print(rid, "reproduced:", same)

H1-win500-gateon-v1-fe051d28 reproduced: True
T4-b8k-units-gateon-v1-34a7b4ae reproduced: True


In [12]:
import inspect
assert list(inspect.signature(Pipeline.run).parameters) == ["self", "question_id", "text"]
labels = {q["question_id"]: q["engine_type"] for q in questions}
for m in ("H1", "F1", "T1", "X1"):
    assert Pipeline(method_config(m, "units", True, PARAMS_VERSION, prompts=prompts), res, labels=labels).labels is None
label_fields = ("question_type", "engine_type", "gold_kind", "difficulty", "mentions_section_id", "gold", "answer")
src = "".join(inspect.getsource(__import__(f"csrag.retrievers.{m}", fromlist=["x"]))
              for m in ("agentic", "structural", "rerank", "fusion", "query_llm", "qdrant_search"))
print("label field names in retriever code:", [f for f in label_fields if f'["{f}"]' in src or f"['{f}']" in src])
print("only F0 reads labels: OK")

label field names in retriever code: ['engine_type']
only F0 reads labels: OK


In [13]:
bad = []
for rid in runs.list_runs(params_version=PARAMS_VERSION):
    cfg = runs.load_config(rid)["config"]
    ex = cfg.get("expansion") or {}
    if ex.get("rule") not in ("refs", "parent+refs", "agent"):
        continue
    h = runs.load_hits(rid)
    h = h[h["in_context"]]
    if (~h["hop"].isin([0, 1])).any():
        bad.append((rid, "hop > 1"))
    for qid, g in h.groupby("question_id"):
        hop0 = set(g.loc[g["hop"] == 0, "unit_id"])
        src_ = set(g.loc[g["hop"] == 1, "source"])
        if not src_ <= hop0:
            bad.append((rid, qid, src_ - hop0))
print(len(bad), "one-hop violations", bad[:5])

0 one-hop violations []


In [14]:
have = set(runs.list_runs(params_version=PARAMS_VERSION))
missing = [runs.run_id_of(c) for c in configs if runs.run_id_of(c) not in have]
print(f"{len(configs) - len(missing)} of {len(configs)} {PARAMS_VERSION} runs present; missing: {missing[:10]}")
pd.Series([runs.load_config(r)["retriever"] for r in have]).value_counts().sort_index().rename("runs").to_frame().T

110 of 110 v1 runs present; missing: []


,C1,D1,F0,F1,H1,H2,L1,M1,Q1,Q2,S1,T0,T1,T2,T2+T3,T3,T4,X1,X2
runs,4,4,4,4,4,4,4,4,4,4,4,12,4,12,12,12,6,4,4


## v2: the one revision (plan 4.8)

After reading **all** v1 metrics in notebook 09, revise the parameters **once**, by hand, in `src/csrag/retrievers/base.py`:

```python
PARAMS["v2"] = revise("v1", gate_a=GateParams(tau=..., delta=..., min_keep=..., max_keep=...), ...)
```

Then set `PARAMS_VERSION = "v2"` and run phase A (notebook 07, GPU encoder, vLLM stopped) and phase B (this notebook) again. Both versions are kept: v1 is the a-priori result, v2 the revised one (chosen after seeing v1 on the same questions).